# Database API Examples
This notebook demonstrates connection management, structural operations, CRUD functionality, and advanced query execution for the Database Abstraction Framework using Postgres SQL.

In [ ]:
from datetime import datetime
from Library.Dataframe import pl
from Library.Database import PostgresDatabaseAPI, PrimaryKey, ForeignKey, QueryAPI

# Administrative Operations

### Create & Check Existence
Admin access is required to create databases. Use `admin=True`.

In [ ]:
with PostgresDatabaseAPI(admin=True) as admin_db:
    if not admin_db.exists(database="example_db"):
        admin_db.create(database="example_db")
    print(f"Database exists: {admin_db.exists(database='example_db')}")

### List Catalogs
Use `.list()` to fetch the structural catalog (Databases, Schemas, Tables).

In [ ]:
with PostgresDatabaseAPI(admin=True) as admin_db:
    catalog_df = admin_db.list(database="example_db")
    display(catalog_df)

### Refactor (Rename)
Structures can be renamed on the fly.

In [ ]:
with PostgresDatabaseAPI(admin=True) as admin_db:
    admin_db.refactor(database="example_db", name="example_db_renamed")
    admin_db.refactor(database="example_db_renamed", name="example_db")

# Structural Operations

### Define & Create Tables (Primary & Foreign Keys)
You can define structures using standard Python types (`int`, `str`), Polars types (`pl.Float64`, `pl.Datetime`), or uninitialized Polars types (`type[pl.DataType]`). Wrap types in `PrimaryKey()` or `ForeignKey()` to add constraints.

In [ ]:
db = PostgresDatabaseAPI(database="example_db").connect()

if not db.exists(schema="example_schema"):
    db.create(schema="example_schema")

# 1. Create a Parent Table
supplier_struct = {
    "id": PrimaryKey(int),
    "company": str
}
db.create(schema="example_schema", table="suppliers", structure=supplier_struct)

# 2. Create a Child Table linking to the Parent Table via ForeignKey
structure: dict[str, type | type[pl.DataType] | pl.DataType | PrimaryKey | ForeignKey] = {
    "id": PrimaryKey(int),
    "supplier_id": ForeignKey(int, reference="example_schema.suppliers(id)"),
    "name": str,
    "price": pl.Float64,
    "timestamp": pl.Datetime
}
db.create(schema="example_schema", table="products", structure=structure)

### Diff & Migrate
Check if the local structure differs from the remote database, and migrate it safely if it does.

In [ ]:
# Add a new column to the structure definition
structure["in_stock"] = pl.Boolean

has_difference = db.diff(schema="example_schema", table="products", structure=structure)
print(f"Structure has changed: {has_difference}")

if has_difference:
    db.migrate(schema="example_schema", table="products", structure=structure)

### Alter Columns (Add, Rename, Reorder, Drop)
You can dynamically modify columns without redefining the whole structure.

In [ ]:
db.add(schema="example_schema", table="products", structure={"temp_col": int})
db.rename(schema="example_schema", table="products", column="temp_col", name="temporary_col")
db.reorder(schema="example_schema", table="products", columns=["id", "supplier_id", "name", "in_stock", "price", "timestamp", "temporary_col"])
db.drop(schema="example_schema", table="products", column="temporary_col")

### Search Schema
Search across the catalog for specific column names or rows.

In [ ]:
search_df = db.search(schema="example_schema", column="price")
display(search_df)
db.disconnect()

# CRUD Operations

### Insert & Select
Data can be inserted as dictionaries, tuples, or DataFrames.

In [ ]:
with PostgresDatabaseAPI(database="example_db", schema="example_schema", table="products") as api:
    api._STRUCTURE_ = structure
    
    # Insert a parent supplier first due to ForeignKey constraint
    api.insert(table="suppliers", data=[{"id": 99, "company": "Farm Corp"}])
    
    # Insert products
    data = [
        {"id": 1, "supplier_id": 99, "name": "Apple", "price": 1.50, "timestamp": datetime.now(), "in_stock": True},
        {"id": 2, "supplier_id": 99, "name": "Banana", "price": 0.75, "timestamp": datetime.now(), "in_stock": True}
    ]
    api.insert(data=data)
    
    display(api.select())

### Update & Remove
Target specific records using `condition` and query variables.

In [ ]:
with PostgresDatabaseAPI(database="example_db", schema="example_schema", table="products") as api:
    api.update(data=[{"id": 2, "price": 0.80}], condition="id = :id:")
    
    api.remove(condition="id = :id:", parameters={"id": 2})
    
    display(api.select(order="id"))

### Upsert (Insert or Update)
Upsert will automatically infer the conflict `key` from the `PrimaryKey` defined in `_STRUCTURE_` if you don't explicitly pass one.

In [ ]:
with PostgresDatabaseAPI(database="example_db", schema="example_schema", table="products") as api:
    api._STRUCTURE_ = structure
    
    api.upsert(data=[
        {"id": 1, "supplier_id": 99, "name": "Apple", "price": 1.60, "timestamp": datetime.now(), "in_stock": True},  # Existing (Updated)
        {"id": 3, "supplier_id": 99, "name": "Orange", "price": 1.20, "timestamp": datetime.now(), "in_stock": False} # New (Inserted)
    ])
    
    display(api.select(columns=["id", "name", "price"], order="id"))

# Custom Queries & Execution

### executeone, executemany, and execute
Leverage the `QueryAPI` to build robust queries with `::interpolation::` and `:named:` or `:?:` positional bindings.

In [ ]:
with PostgresDatabaseAPI(database="example_db", schema="example_schema", table="products") as api:
    insert_query = QueryAPI("INSERT INTO ::schema::.::table:: (id, supplier_id, name, price, in_stock) VALUES (:id:, :supplier_id:, :name:, :price:, :in_stock:)")
    
    # 1. executeone(): Runs a single time with kwargs
    api.executeone(insert_query, id=4, supplier_id=99, name="Mango", price=2.0, in_stock=True)
    
    # 2. executemany(): Runs in a batch via list
    batch_data = [
        {"id": 5, "supplier_id": 99, "name": "Grapes", "price": 2.5, "in_stock": True},
        {"id": 6, "supplier_id": 99, "name": "Peach", "price": 3.0, "in_stock": False}
    ]
    api.executemany(insert_query, batch_data)
    
    # 3. execute(): Auto-delegates based on input shape
    api.execute(insert_query, {"id": 7, "supplier_id": 99, "name": "Plum", "price": 1.75, "in_stock": True})

### fetchone, fetchmany, fetchall
Retrieve chunks of a query execution result.

In [ ]:
with PostgresDatabaseAPI(database="example_db", schema="example_schema", table="products") as api:
    select_query = QueryAPI("SELECT * FROM ::schema::.::table:: ORDER BY id")
    
    # 1. Fetch One
    print("\nFetch One Result:")
    display(api.executeone(select_query).fetchone())
    
    # 2. Fetch Many
    print("\nFetch Many (2) Results:")
    display(api.executeone(select_query).fetchmany(n=2))
    
    # 3. Fetch All
    print("\nFetch All Results:")
    display(api.executeone(select_query).fetchall())

# Session Management & Cleanup

### sessions, kill, and delete
Observe active sessions, terminate them, and drop the database.

In [ ]:
with PostgresDatabaseAPI(admin=True) as admin_db:
    sessions_df = admin_db.sessions(database="example_db")
    print("Active Sessions before kill:")
    display(sessions_df)
    
    admin_db.kill(database="example_db")
    admin_db.delete(database="example_db")
    print("Cleanup Complete.")